# Countdown GRPO from the SFT start: format control vs format + judge

Trains Qwen model for Countdown using the judge trained by Consistency_Qwen_Judge_Countdown. 

In [1]:
# [config] Everything you choose lives here.
MODELS = {   # key: HF id, chat-template kwargs (Qwen3: thinking off, as the SFT and the screens)
    "qwen3-0.6b": ("Qwen/Qwen3-0.6B", {"enable_thinking": False}),
    "qwen3-1.7b": ("Qwen/Qwen3-1.7B", {"enable_thinking": False}),
}
MODEL_KEY = "qwen3-0.6b"
MODEL, CHAT_KWARGS = MODELS[MODEL_KEY]
SEED = 0
TRAIN_SEED = 0                   # the training RNG and the colocated vLLM engine's seed ([build]); 0 = every run so far
# SEED0_STEP0: a seed-0 run's step-0 rollouts; with TRAIN_SEED != 0, [train] stops after step 0 if they repeat them
SEED0_STEP0 = "/content/drive/MyDrive/Consistency/countdown/grpo/qwen3-0.6b-sft-judge-w1-median/step_00000.txt"
SMOKE = False                    # True: every cell end to end on a few puzzles, 2 GRPO steps per arm (-smoke)
# S: Consistency_SFT_Countdown's run (same settings there)
SFT_RUN = f"{MODEL.split('/')[-1]}-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500"
TEMPERATURE = 0.9
MAX_COMPLETION_LENGTH = 1024     # cap, as the Llama runs (the judge's data was generated at 2048)
# The judge: Consistency_Qwen_Judge_Countdown's DPO adapter, trained on "empty" corruptions
JUDGE_RUN = ("judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/"
             "dpo_empty_b0.1_lr2e-05_ep2")
N_CORRUPT_4, N_CORRUPT_3 = 1000, 1000            # Consistency_Qwen_Judge_Countdown's corruption puzzles: kept out of the training puzzles
MAX_SOLUTION_TOKENS = 128        # the judge scores solution + answer of at most this many tokens; else missing
# GRPO arms: "format" (control) and "judge" (format + judge)
ARMS = ("judge",)       # run in this order; each skipped when trained to MAX_STEPS
JUDGE_REDUCTION = "sum"
MAX_STEPS = 150
PUZZLES_PER_STEP, NUM_GENERATIONS = 8, 16        # 4 four-number + 4 three-number puzzles x 16 = 256 per step
LEARNING_RATE, WARMUP_STEPS = 2e-5, 20
KL_BETA = 0.02
LOSS_TYPE, SCALE_REWARDS, MAX_GRAD_NORM = "dapo", False, 0.2
VLLM_IS_MODE, VLLM_IS_CLIP = "token_truncate", 2.0
ADAM_BETA2, WEIGHT_DECAY = 0.99, 0.0
MICRO_BATCH = 8
JUDGE_WEIGHT_SCALE = 1.0
EXPRESSION_REWARD = True
SAVE_STEPS, DUMP_EVERY = 25, 25
VLLM_GPU_MEMORY_UTILIZATION = 0.5
CURVE_EVERY, CURVE_N, CURVE_K = 25, 100, 4      # held-out curve: CURVE_N per size, greedy x 1 + sampled x CURVE_K
N_GRADE, GRADE_SAMPLES = 200, 8                  # S's grade puzzles per size, as S's grade file
GRADE_ALSO = (50,100,)                           # checkpoints graded besides the final policy
UNASSIGN_WHEN_DONE = True                        # [release]
if SMOKE:
    MAX_STEPS, SAVE_STEPS, DUMP_EVERY = 2, 1, 1
    CURVE_EVERY, CURVE_N, CURVE_K, N_GRADE, GRADE_ALSO = 1, 4, 2, 4, (1,)

DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
_sm = "-smoke" if SMOKE else ""
S_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN}"
S_ADAPTER = f"{S_DIR}/adapter"
S_GRADE = f"{S_DIR}/grade_n200x8_t{TEMPERATURE:g}_cap{MAX_COMPLETION_LENGTH}_seed{SEED}{_sm}/gen_sft.csv"   # [judge] makes it if missing
GRADE_FILES = {n: f"{DRIVE}/countdown/grpo/olmo3-7b-format50-cap1024-seed0/grade/gen_Olmo-3-7B-Instruct-format50_user_n{n}.csv"
               for n in (4, 3)}          # the grade puzzles (S's were drawn from these)
JUDGE_DIR = f"{S_DIR}/{JUDGE_RUN}"
JUDGE_PATH = f"{JUDGE_DIR}/adapter"

CALIB_PATH = f"{JUDGE_DIR}/calibration_{JUDGE_REDUCTION}_missmed_S_{S_GRADE.split('/')[-2]}{_sm}.json"   # made by [judge]
# The run folder is {model}-sft-{arm}-{RUN_TAG}; the settings live in its run.json. Rerunning a folder with other
# settings asks first ([train]); raising MAX_STEPS continues the run (the puzzle file grows, the steps so far are kept).
RUN_TAG = "w1-median"
_short = MODEL_KEY + "-sft"
RUN_IDS = {arm: f"{_short}-{arm}-{RUN_TAG}{_sm}" for arm in ARMS}
RUN_DIRS = {arm: f"{DRIVE}/countdown/grpo/{r}" for arm, r in RUN_IDS.items()}
CURVE_S = f"{S_DIR}/curve_S_n34x{CURVE_N}_greedy_pass{CURVE_N}x{CURVE_K}_t{TEMPERATURE:g}_cap{MAX_COMPLETION_LENGTH}_seed{SEED}{_sm}"
TRAIN_PUZZLES_CSV = f"{S_DIR}/grpo_train_puzzles_n34_x{PUZZLES_PER_STEP}_seed{SEED}{_sm}.csv"   # both arms; grows with MAX_STEPS
assert set(ARMS) <= {"format", "judge"} and JUDGE_REDUCTION in ("mean", "sum")
assert (PUZZLES_PER_STEP * NUM_GENERATIONS) % MICRO_BATCH == 0 and PUZZLES_PER_STEP % 2 == 0
assert all(s % SAVE_STEPS == 0 and s < MAX_STEPS for s in GRADE_ALSO)
print(f"MODEL={MODEL} {CHAT_KWARGS}\nS={S_ADAPTER}\nJUDGE_DIR={JUDGE_DIR}\n" + "\n".join(f"{a}: {d}" for a, d in RUN_DIRS.items()))


MODEL=Qwen/Qwen3-0.6B {'enable_thinking': False}
S=/content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/adapter
JUDGE_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/dpo_empty_b0.1_lr2e-05_ep2
judge: /content/drive/MyDrive/Consistency/countdown/grpo/qwen3-0.6b-sft-judge-w1-median


In [2]:
# [drive + HF] Mount Drive, HF login (Colab secret HF_TOKEN), and the checks that must fail before any GPU work.
import os, glob, shutil, json
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
for f in ("countdown.py", "prompts.py", "rewards.py"):
    assert os.path.exists(f"{CODE_DIR}/{f}"), f"missing {CODE_DIR}/{f}"
assert os.path.exists(f"{S_ADAPTER}/adapter_model.safetensors"), f"no S adapter in {S_ADAPTER}"
for f in GRADE_FILES.values():
    assert os.path.exists(f), f"missing {f}"
assert os.path.exists(f"{JUDGE_PATH}/adapter_model.safetensors"), f"no judge in {JUDGE_PATH}: run Consistency_Qwen_Judge_Countdown first"
for d in RUN_DIRS.values():
    os.makedirs(d, exist_ok=True)
print("judge:", JUDGE_PATH, "| calibration:", "exists" if os.path.exists(CALIB_PATH) else "made by [judge]")
def trained_to(d):
    """The step a run folder's final adapter was trained to (0: none)."""
    _r = json.load(open(f"{d}/run.json")) if os.path.exists(f"{d}/run.json") else {}
    ok = _r.get("status") == "trained" and os.path.exists(f"{d}/adapter/adapter_model.safetensors")
    return int(_r["config"]["MAX_STEPS"]) if ok else 0

for a, d in RUN_DIRS.items():
    _c = glob.glob(f"{d}/checkpoint-*")
    print(f"{a}:", "done" if trained_to(d) == MAX_STEPS
          else (f"resumes from {max(_c, key=lambda p: int(p.rsplit('-', 1)[1]))}" if _c else "from S"))


Mounted at /content/drive
judge: /content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/dpo_empty_b0.1_lr2e-05_ep2/adapter | calibration: exists
judge: from S


In [3]:
# [install] Once per fresh runtime, no restart needed: skipped when TRL >= 1 and vLLM are
# installed. pip runs as a subprocess, not %pip, so Colab shows no "restart the session" prompt; Pillow is pinned to the
# version Colab already imported. torchao is removed (peft refuses Colab's 0.10).
import importlib.metadata as _md, importlib.util, subprocess, sys
try:
    _ready = int(_md.version("trl").split(".")[0]) >= 1 and importlib.util.find_spec("vllm") is not None
except _md.PackageNotFoundError:
    _ready = False
if _ready:
    print(f"[install] skipped: trl {_md.version('trl')} and vllm {_md.version('vllm')} are installed")
else:
    _pin = [f"pillow=={_md.version('pillow')}"] if importlib.util.find_spec("PIL") else []
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl[vllm]", "peft", *_pin], check=True)
    print(f"[install] done: trl {_md.version('trl')}, vllm {_md.version('vllm')}" + (f", {_pin[0]} kept" if _pin else ""))
if importlib.util.find_spec("torchao"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")


[install] done: trl 1.18.0, vllm 0.31.0, pillow==11.3.0 kept
[install] removed torchao


In [4]:
# [vLLM setup] Before anything touches the GPU. (No expandable_segments: with it vLLM's CUDA-graph capture failed,
# Consistency_MATH_Finish_Probe, 2026-11-12.)
import os, sys, subprocess
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the vLLM engine in this process
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):   # Colab's stdout/stderr have no OS file descriptor
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)
import importlib.metadata as md
for p in ["torch", "transformers", "trl", "peft", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")


name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB

torch          2.13.0
transformers   5.17.0
trl            1.18.0
peft           0.21.3
datasets       4.8.5
vllm           0.31.0


In [5]:
# [setup] Tokenizer (CHAT_KWARGS on every template call), the Countdown code (Drive), the labels (the screen's / S's
# grade file's) and the judge's text views.
import re, json, importlib, math, time
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
assert tok.pad_token is not None and tok.pad_token_id != tok.eos_token_id, tok.pad_token

def chat_text(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_KWARGS)

def chat_ids(messages):
    return tok(chat_text(messages), add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)
import countdown, prompts, rewards as R
for _m in (countdown, prompts, R):
    importlib.reload(_m)
from countdown import check_answer
from prompts import countdown_prompt, judge_messages
tag, completion_text = R.tag, R.completion_text
_b = lambda d, c: d[c].astype(str) == "True"
_rd = lambda p: pd.read_csv(p, keep_default_na=False, dtype={c: str for c in ("completion", "reasoning", "solution", "answer",
                                                                              "generated", "text")})

def label(df):
    """As Consistency_SFT_Countdown's [grade] (the screen's): blocks, format_ok, answered, steps_parsed, correct, diagnosis."""
    df = df.copy()
    texts, nums, targets = list(df.completion), [json.loads(x) for x in df.numbers], list(df.target)
    for b in ("reasoning", "solution", "answer"):
        df[b] = [R.tag(t, b) or "" for t in texts]
    df["format_ok"] = [bool(re.search(r"<reasoning>.*?</reasoning>\s*<solution>.*?</solution>\s*<answer>.*?</answer>",
                                      t, re.DOTALL)) for t in texts]
    df["answered"] = df.answer.str.strip() != ""
    df["steps_parsed"] = [bool(R.parse_steps(s)) for s in df.solution]
    df["correct"] = np.array(R.correctness_reward_func(None, texts, nums, targets)) > 0
    df["diagnosis"] = [R.diagnose(a or None, n, t, tr) for a, n, t, tr in zip(df.answer, nums, targets, df.truncated)]
    return df

def read_graded(path):
    d = _rd(path)
    for c in ("finished", "truncated", "format_ok", "answered", "steps_parsed", "correct"):
        d[c] = _b(d, c)
    d["puzzle"], d["n_numbers"] = d.puzzle.astype(int), d.n_numbers.astype(int)
    return d

def stats(d):
    """As the SFT grade's table (cap MAX_COMPLETION_LENGTH)."""
    fin = d.finished
    ans, ok = fin & d.answered, fin & d.correct
    g = d.assign(aw=ans & ~ok, ok=ok).groupby("puzzle")
    return {"n": len(d), "accuracy": ok.mean(), "format ok": (fin & d.format_ok).mean(),
            "solution parses": (fin & d.steps_parsed).mean(), "truncated": 1 - fin.mean(),
            "answered wrong": (ans & ~ok).mean(), "correct among answered": ok.sum() / max(ans.sum(), 1),
            "tokens (median, finished)": d.n_tokens[fin].median(),
            "puzzles: correct + answered wrong": int((g.ok.any() & g.aw.any()).sum()), "pass@k": g.ok.any().mean()}

def scorable(sol, ans):
    """Scored by the GRPO reward: a solution of at most MAX_SOLUTION_TOKENS tokens and an answer."""
    return (isinstance(sol, str) and bool(sol.strip()) and isinstance(ans, str) and bool(ans.strip())
            and len(tok(sol, add_special_tokens=False)["input_ids"]) <= MAX_SOLUTION_TOKENS)

judge_text = lambda sol, ans: f"{sol.strip()}\nAnswer: {ans.strip()}"

def end_of_turn_id(tok):
    s = tok.apply_chat_template([{"role": "user", "content": "Q"}, {"role": "assistant", "content": "XYZXYZ"}],
                                tokenize=False, **CHAT_KWARGS)
    eot = tok(s.split("XYZXYZ", 1)[1], add_special_tokens=False)["input_ids"][0]
    assert tok.convert_ids_to_tokens(eot) in tok.all_special_tokens, tok.convert_ids_to_tokens(eot)
    return eot

EOT = end_of_turn_id(tok)
_p = chat_text([{"role": "user", "content": "x"}])
if CHAT_KWARGS.get("enable_thinking") is False:
    assert _p.rstrip().endswith("</think>"), "thinking-off template should end with an empty think block"
print("tokenizer:", MODEL, "| pad", tok.pad_token, "| eos", tok.eos_token, "| end of turn", tok.decode([EOT]),
      "| generation prompt ends", repr(_p[-40:]))


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

tokenizer: Qwen/Qwen3-0.6B | pad <|endoftext|> | eos <|im_end|> | end of turn <|im_end|> | generation prompt ends '|im_start|>assistant\n<think>\n\n</think>\n\n'


In [6]:
# [puzzles] From Jiayi-Pan/Countdown-Tasks-3to4, disjoint by construction:
#   grade       S's grade puzzles (the grade files: 200 four- + 200 three-number of the test draw SEED), first N_GRADE
#   corruption  Consistency_Qwen_Judge_Countdown's (draw SEED + 6), only kept out of the training and curve puzzles
#   training    TRAIN_PUZZLES_CSV (both arms), per step PUZZLES_PER_STEP / 2 four- then as many three-number puzzles.
#               New file: draw SEED + 8, MAX_STEPS steps, apart from all of the above (whole draws) and the corruption
#               draw. A file with fewer steps than MAX_STEPS keeps its steps and grows by draw SEED + 12 (apart from
#               everything, the curve too); one with more is used up to MAX_STEPS.
#   curve       draw SEED + 10: CURVE_N of each size, apart from everything above (the file's first training steps)
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])
_seeded = lambda seed, size: _tz.select(np.random.default_rng(seed).choice(len(_tz), size=min(len(_tz), size), replace=False))

GRADE = {}
for n, f in GRADE_FILES.items():
    g = pd.read_csv(f, keep_default_na=False).drop_duplicates("puzzle").sort_values("puzzle").head(N_GRADE)
    GRADE[n] = [{"puzzle": int(r.puzzle), "numbers": json.loads(r.numbers), "target": int(r.target)} for r in g.itertuples()]
_draw0 = _seeded(SEED, 20 * 200)
assert [_key(p) for p in GRADE[4]] == [_key(_as(r)) for r in _draw0 if len(r["nums"]) == 4][1:N_GRADE + 1]

_no = {_key(_as(r)) for seed, size in ((SEED, 20 * 200), (SEED + 1, 6 * (4000 + 100)), (SEED + 2, 4 * (4000 + 100)),
                                        (SEED + 3, 4 * 3900)) for r in _seeded(seed, size)}
_want, _got = {4: N_CORRUPT_4, 3: N_CORRUPT_3}, {4: [], 3: []}
for r in _seeded(SEED + 6, 10 * (N_CORRUPT_4 + N_CORRUPT_3)):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(_got[n]) < _want[n]:
        _no.add(_key(p))
        _got[n].append(p)
assert all(len(_got[n]) == _want[n] for n in _want), {n: len(v) for n, v in _got.items()}
corrupt_puzzles = _got[4] + _got[3]
_no |= {_key(_as(r)) for r in _seeded(SEED + 6, 10 * (N_CORRUPT_4 + N_CORRUPT_3))}

_half = PUZZLES_PER_STEP // 2
_from_df = lambda df: [{"numbers": json.loads(r.numbers), "target": int(r.target)} for r in df.itertuples()]
if os.path.exists(TRAIN_PUZZLES_CSV):
    _train = _from_df(pd.read_csv(TRAIN_PUZZLES_CSV))
    assert len(_train) % PUZZLES_PER_STEP == 0 and all(
        len(p["numbers"]) == (4 if i % PUZZLES_PER_STEP < _half else 3) for i, p in enumerate(_train)), TRAIN_PUZZLES_CSV
else:
    _need = MAX_STEPS * _half
    _tr = {4: [], 3: []}
    for r in _seeded(SEED + 8, 40 * _need):
        p = _as(r)
        n = len(p["numbers"])
        if _key(p) not in _no and len(_tr[n]) < _need:
            _no.add(_key(p))
            _tr[n].append(p)
    assert all(len(v) == _need for v in _tr.values()), {n: len(v) for n, v in _tr.items()}
    _train = [p for s in range(MAX_STEPS) for n in (4, 3) for p in _tr[n][s * _half:(s + 1) * _half]]
_saved_steps = len(_train) // PUZZLES_PER_STEP
_write = _saved_steps < MAX_STEPS or not os.path.exists(TRAIN_PUZZLES_CSV)
_no |= {_key(p) for p in _train}
CURVE = {4: [], 3: []}
for r in _seeded(SEED + 10, 40 * CURVE_N):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(CURVE[n]) < CURVE_N:
        _no.add(_key(p))
        CURVE[n].append(p)
assert all(len(v) == CURVE_N for v in CURVE.values())
if _saved_steps < MAX_STEPS:                     # grow the file: the extra steps only
    _need = (MAX_STEPS - _saved_steps) * _half
    _tr = {4: [], 3: []}
    for r in _seeded(SEED + 12, 40 * _need):
        p = _as(r)
        n = len(p["numbers"])
        if _key(p) not in _no and len(_tr[n]) < _need:
            _no.add(_key(p))
            _tr[n].append(p)
    assert all(len(v) == _need for v in _tr.values()), {n: len(v) for n, v in _tr.items()}
    _train += [p for s in range(MAX_STEPS - _saved_steps) for n in (4, 3) for p in _tr[n][s * _half:(s + 1) * _half]]
    print(f"[puzzles] training file: {_saved_steps} steps kept, {MAX_STEPS - _saved_steps} drawn")
_train = _train[:MAX_STEPS * PUZZLES_PER_STEP]
_tdf = pd.DataFrame([{"pid": i, "n": len(p["numbers"]), "numbers": json.dumps(p["numbers"]), "target": p["target"]}
                     for i, p in enumerate(_train)])
if _write:
    _tdf.to_csv(TRAIN_PUZZLES_CSV, index=False)
train_dataset = Dataset.from_list([{"prompt": countdown_prompt(p["numbers"], p["target"]), "numbers": p["numbers"],
                                    "target": p["target"], "pid": i, "n": len(p["numbers"])} for i, p in enumerate(_train)])
_gk = {_key(p) for v in GRADE.values() for p in v}
assert not _gk & ({_key(p) for p in _train} | {_key(p) for p in corrupt_puzzles} | {_key(p) for v in CURVE.values() for p in v})
print(f"grade {({n: len(v) for n, v in GRADE.items()})}, corruption (excluded) {len(corrupt_puzzles)}, training {len(train_dataset)} ({MAX_STEPS} steps x {_half} + {_half}), "
      f"curve {({n: len(v) for n, v in CURVE.items()})}")


README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.85MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

grade {4: 200, 3: 200}, corruption (excluded) 2000, training 1200 (150 steps x 4 + 4), curve {4: 100, 3: 100}


In [7]:
# [vLLM sync fix] TRL syncs LoRA to vLLM by merging dW into the training model's bf16 W
# in place, then unmerging; each add/subtract rounds, so the frozen W drifts. This version streams bf16(W + dW) built in
# a temporary tensor (one rounding) and never touches the training model. Only active adapters are merged.
import torch
from accelerate.utils import is_peft_model
from peft.tuners.lora import LoraLayer
from trl.generation.vllm_generation import VLLMGeneration

if not hasattr(VLLMGeneration, "_trl_iter_named_params"):
    VLLMGeneration._trl_iter_named_params = VLLMGeneration._iter_named_params

def _iter_named_params_out_of_place(self):
    model = self.model
    if not is_peft_model(model) or self.accelerator.num_processes > 1:
        yield from self._trl_iter_named_params()
        return
    lora_layers = {name: m for name, m in model.named_modules() if isinstance(m, LoraLayer)}
    n_merged = 0
    for name, param in model.named_parameters():
        vllm_name = name.removeprefix("base_model.model.").replace(".base_layer", "")
        if model.prefix in vllm_name or "original_module" in vllm_name:
            continue
        vllm_name = self._fix_param_name_to_vllm(vllm_name, extra_prefixes=["modules_to_save.default."])
        layer = lora_layers.get(name.rsplit(".base_layer.", 1)[0]) if name.endswith(".base_layer.weight") else None
        if layer is None:
            yield vllm_name, param.data
            continue
        assert not layer.merged, f"{name}: adapter is merged in the training model"
        with torch.no_grad():
            merged = param.data.float()
            for adapter in layer.active_adapters:
                if adapter in layer.lora_A:
                    merged += layer.get_delta_weight(adapter).float()
            merged = merged.to(param.dtype)
        n_merged += 1
        yield vllm_name, merged
        del merged
    if not getattr(self, "_out_of_place_reported", False):
        print(f"[vLLM sync fix] streamed {n_merged} LoRA layers as bf16(W + dW); training weights untouched")
        self._out_of_place_reported = True

VLLMGeneration._iter_named_params = _iter_named_params_out_of_place
print("VLLMGeneration._iter_named_params patched")


VLLMGeneration._iter_named_params patched


In [8]:
# [rewards] S's format rewards (xmlcount, soft / strict format, expression); the judge (reward weight set per arm in
# [train]: calibrated in the judge arm, 0 in the format arm) scores, under its frozen adapter, the judge prompt +
# "solution\nAnswer: answer" + end of turn when both parse and the solution has <= MAX_SOLUTION_TOKENS tokens; else
# missing = JUDGE_MISSING (set by [judge]). Logged with weight 0: correctness, the judge's mean and sum, S's monitors;
# the step monitor prints every step and dumps every DUMP_EVERY to the arm's RUN_DIR.
import contextlib
from trl.trainer.utils import selective_log_softmax
CUR = {"arm": None, "run_dir": None}            # the arm [train] runs
JUDGE_MISSING = float("nan")                    # set by [judge]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>") == 1:
        count += 0.125
    if text.count("</reasoning>") == 1:
        count += 0.125
    if text.count("<solution>") == 1:
        count += 0.005
    if text.count("</solution>") == 1:
        count += 0.005
    if text.count("<answer>") == 1:
        count += 0.125
    if text.count("</answer>") == 1:
        count += 0.125
    if "</answer>" in text:
        count -= (len(text.split("</answer>", 1)[1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    return [count_xml(completion_text(c)) for c in completions]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>[\s\S]*?</reasoning>\s*<solution>[\s\S]*?</solution>\s*<answer>[\s\S]*?</answer>"
    return [0.5 if re.match(pattern, completion_text(c)) else 0.0 for c in completions]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>\s*.*?\s*</reasoning>\s*<solution>\s*.*?\s*</solution>\s*<answer>\s*.*?\s*</answer>"
    return [0.5 if re.search(pattern, completion_text(c), re.DOTALL) else 0.0 for c in completions]

FORMAT_FUNCS = [xmlcount_reward_func, soft_format_reward_func, strict_format_reward_func] + (
    [R.expression_reward_func] if EXPRESSION_REWARD else [])

def format_total(texts):
    """The summed format reward of completion texts (what the format arm trains on)."""
    cs = [[{"role": "assistant", "content": t}] for t in texts]
    return np.sum([f(completions=cs) for f in FORMAT_FUNCS], axis=0)

def judge_input(text):
    """The text the judge scores, or None (missing)."""
    s, a = tag(text, "solution"), tag(text, "answer")
    return judge_text(s, a) if scorable(s, a) else None

def _freeze_others(model):
    for n, p in model.named_parameters():        # set_adapter makes the active adapter trainable: only the policy is
        if "lora_" in n and ".default." not in n:
            p.requires_grad_(False)

@contextlib.contextmanager
def scorer_weights(model, adapter):
    """adapter: "judge", "ref" (= S, frozen) or None (the untrained model: adapters off); then the policy again."""
    if adapter is None:
        with model.disable_adapter():
            yield
        return
    model.set_adapter(adapter)
    try:
        yield
    finally:
        model.set_adapter("default")
        _freeze_others(model)

def judge_scores(model, items, adapter="judge", chunk=16):
    """(mean, sum) arrays over items = [(numbers, target, text or None)], NaN for None: per-token log-probs of the text +
    end of turn after the judge prompt."""
    means, sums = np.full(len(items), np.nan), np.full(len(items), np.nan)
    seqs = {i: (chat_ids(judge_messages(list(n), int(t))), tok(x, add_special_tokens=False)["input_ids"] + [EOT])
            for i, (n, t, x) in enumerate(items) if x}
    order = sorted(seqs, key=lambda i: len(seqs[i][0]) + len(seqs[i][1]))
    fwd = getattr(model, "_original_forward", model.forward)
    with torch.no_grad(), scorer_weights(model, adapter):
        for s in range(0, len(order), chunk):
            idx = order[s:s + chunk]
            L = max(len(seqs[i][0]) + len(seqs[i][1]) for i in idx)
            ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, i in enumerate(idx):
                x = seqs[i][0] + seqs[i][1]
                ids[j, :len(x)], attn[j, :len(x)] = torch.tensor(x), 1
            ids, attn = ids.to(model.device), attn.to(model.device)
            per_pos = selective_log_softmax(fwd(input_ids=ids, attention_mask=attn).logits[:, :-1], ids[:, 1:]).float()
            for j, i in enumerate(idx):
                p, t = seqs[i]
                lp = per_pos[j, len(p) - 1:len(p) + len(t) - 1].cpu().numpy()
                means[i], sums[i] = lp.mean(), lp.sum()
            del per_pos
    return means, sums

_LAST = {}

def make_judge_reward(owner):
    def judge(completions, numbers, target, **kwargs):
        assert "judge" in owner.model.peft_config and not math.isnan(JUDGE_MISSING), "run [judge] first"
        items = [(n, t, judge_input(completion_text(c))) for n, t, c in zip(numbers, target, completions)]
        m, s = judge_scores(owner.model, items)
        _LAST.update(id=id(completions), mean=m, sum=s)
        v = m if JUDGE_REDUCTION == "mean" else s
        return [JUDGE_MISSING if math.isnan(x) else float(x) for x in v]
    judge.__name__ = "judge"
    return judge

def _last(key, completions):
    v = _LAST.get(key) if _LAST.get("id") == id(completions) else None
    return [None] * len(completions) if v is None else [None if math.isnan(x) else float(x) for x in v]

def judge_mean(completions, **kwargs):
    return _last("mean", completions)

def judge_sum(completions, **kwargs):
    return _last("sum", completions)

def correct(completions, numbers, target, **kwargs):
    return [float(x > 0) for x in R.correctness_reward_func(None, completions, numbers, target)]

MONITORS = list(R.MONITORS)

def within_puzzle_win_rate(keys, scores, correct):
    """Over (correct, wrong) pairs of the same puzzle: share where the correct one scores higher, ties 1/2."""
    wins = pairs = 0
    keys = np.array(keys)
    for k in set(keys.tolist()):
        s, c = scores[keys == k], correct[keys == k]
        d = s[c][:, None] - s[~c][None, :]
        wins += (d > 0).sum() + 0.5 * (d == 0).sum()
        pairs += d.size
    return (wins / pairs if pairs else float("nan")), int(pairs)

def make_step_monitor(owner):
    def step_monitor(prompts, completions, numbers, target, pid, **kwargs):
        step_monitor_check(completions)
        texts = [completion_text(c) for c in completions]
        cn = np.array(correct(completions, numbers, target)) > 0
        fin = np.array([("</answer>" in t) for t in texts])
        v = np.array(judge_mean(completions) if JUDGE_REDUCTION == "mean" else judge_sum(completions), dtype=float)
        v = np.where(np.isnan(v), JUDGE_MISSING, v)
        rate, pairs = within_puzzle_win_rate(pid, v, cn)
        n_ = np.array([len(x) for x in numbers])
        line = (f"correct {cn.mean():.3f} (4: {cn[n_ == 4].mean():.3f}, 3: {cn[n_ == 3].mean():.3f}) | answered "
                f"{fin.mean():.2f} | correct among answered "
                f"{cn[fin].mean() if fin.any() else float('nan'):.3f} | judge {JUDGE_REDUCTION} {v.mean():.3f}, "
                f"win {rate:.2f} ({pairs} pairs)")
        log = kwargs.get("log_metric")
        if log:
            log("correct_among_finished", float(cn[fin].mean()) if fin.any() else float("nan"))
            if pairs:
                log("judge_win_rate", float(rate))
        step = owner.state.global_step
        print(f"[{CUR['arm']} step {step}] {line}")
        if step % DUMP_EVERY == 0:
            with open(f"{CUR['run_dir']}/step_{step:05d}.txt", "w", encoding="utf-8") as f:
                f.write(f"=== Step {step} ===\n\n")
                for i, p in enumerate(prompts):
                    f.write(f"### Sample {i + 1}\n[Scores] puzzle={pid[i]} correct={bool(cn[i])} "
                            f"judge_{JUDGE_REDUCTION}={v[i]:.4f}\n"
                            f"[Prompt]\n{p[-1]['content'].strip()}\n\n[Response]\n{texts[i].strip()}\n\n"
                            + "=" * 80 + "\n\n")
        return [0.0] * len(completions)
    step_monitor.__name__ = "step_monitor"
    return step_monitor

def step_monitor_check(completions):
    """Qwen3 thinking off: no training completion may open <think> (TRL's chat_template_kwargs applied)."""
    if CHAT_KWARGS.get("enable_thinking") is False:
        _t = sum("<think>" in completion_text(c) for c in completions)
        assert _t == 0, f"{_t} training completions open <think>: TRL did not apply enable_thinking=False"
    return True

REWARD_FUNCS_TRAIN = ([(f, 1.0) for f in FORMAT_FUNCS] + [("judge", 0.0)]
                      + [(correct, 0.0), (judge_mean, 0.0), (judge_sum, 0.0)] + [(f, 0.0) for f in MONITORS] + [("step_monitor", 0.0)])
JUDGE_IDX = [f for f, _ in REWARD_FUNCS_TRAIN].index("judge")
print("reward functions:", [f if isinstance(f, str) else f.__name__ for f, _ in REWARD_FUNCS_TRAIN])


reward functions: ['xmlcount_reward_func', 'soft_format_reward_func', 'strict_format_reward_func', 'expression_reward_func', 'judge', 'correct', 'judge_mean', 'judge_sum', 'uses_given_numbers', 'made_up_numbers', 'solution_arithmetic_ok', 'solution_reaches_target', 'answer_matches_solution', 'stopped', 'gave_up', 'caught_claims', 'step_monitor']


In [9]:
# [build] The policy (S's adapter "default", trainable), the GRPO config and trainer (TRL copies S to a frozen "ref"
# adapter: the KL reference, and S's own scorer in [judge]); its vLLM engine samples everything in this notebook.
# Checks: adapters, the trainer's prompt tokens = chat_ids (thinking off), vLLM vs trainer log-probs. The judge adapter
# is added in [judge].
import csv, tempfile
from datetime import datetime
from transformers import AutoModelForCausalLM, TrainerCallback
from peft import PeftModel
from trl import GRPOTrainer, GRPOConfig
from vllm import SamplingParams

class CSVLoggerCallback(TrainerCallback):
    """Every log row to a CSV; the header grows when new keys appear."""
    def __init__(self, path):
        self.path, self.fields = path, None
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if os.path.exists(path) and os.path.getsize(path):
            with open(path, newline="") as f:
                self.fields = next(csv.reader(f))
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        logs = {"step": state.global_step, **logs}
        if self.fields is None:
            self.fields = list(logs)
            with open(self.path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()
        new = [k for k in logs if k not in self.fields]
        if new:
            self.fields += sorted(new)
            rows = list(csv.DictReader(open(self.path, newline="")))
            fd, tmp = tempfile.mkstemp(dir=os.path.dirname(self.path))
            os.close(fd)
            with open(tmp, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=self.fields)
                w.writeheader()
                w.writerows(rows)
            shutil.move(tmp, self.path)
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(logs)

class JudgeGRPOTrainer(GRPOTrainer):
    def log(self, logs, start_time=None):
        super().log(logs, start_time)
        if "reward" in logs and "train_runtime" not in logs:
            v = lambda k: float("nan") if logs.get(k) is None else float(logs[k])
            print(f"[{CUR['arm']} {self.state.global_step}] reward={v('reward'):.3f}, no-gradient groups="
                  f"{v('frac_reward_zero_std'):.2f}, grad_norm={v('grad_norm'):.3f}, kl={v('kl'):.4f}, "
                  f"length={v('completions/mean_length'):.0f}, truncated={v('completions/clipped_ratio'):.3f}, "
                  f"correct={v('rewards/correct/mean'):.3f}, judge {JUDGE_REDUCTION}="
                  f"{v(f'rewards/judge_{JUDGE_REDUCTION}/mean'):.3f}")

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="auto")
model.config.pad_token_id = tok.pad_token_id
model.generation_config.pad_token_id = tok.pad_token_id
model = PeftModel.from_pretrained(model, S_ADAPTER, is_trainable=True)          # "default" = S, continued per arm

COMPLETIONS_PER_STEP = PUZZLES_PER_STEP * NUM_GENERATIONS
training_args = GRPOConfig(
    use_vllm=True, vllm_mode="colocate", vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    vllm_max_model_length=MAX_COMPLETION_LENGTH + 512,
    chat_template_kwargs=CHAT_KWARGS, gradient_checkpointing=True,
    loss_type=LOSS_TYPE, scale_rewards=SCALE_REWARDS, max_grad_norm=MAX_GRAD_NORM, temperature=TEMPERATURE,
    beta=KL_BETA, reward_weights=[w for _, w in REWARD_FUNCS_TRAIN],
    learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_scheduler_type="constant_with_warmup", optim="adamw_torch",
    adam_beta1=0.9, adam_beta2=ADAM_BETA2, weight_decay=WEIGHT_DECAY,
    vllm_importance_sampling_correction=True, vllm_importance_sampling_mode=VLLM_IS_MODE,
    vllm_importance_sampling_clip_max=VLLM_IS_CLIP,
    seed=SEED, shuffle_dataset=False, logging_strategy="steps", logging_steps=5,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=None, output_dir=RUN_DIRS[ARMS[0]], report_to=[],
    bf16=True, per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=COMPLETIONS_PER_STEP // MICRO_BATCH,
    num_generations=NUM_GENERATIONS, max_completion_length=MAX_COMPLETION_LENGTH, max_steps=MAX_STEPS,
)
_bound = {}

def _late(name):
    def f(**kwargs):
        return _bound[name](**kwargs)
    f.__name__ = name
    return f

# The rollout seed: TRL builds its colocated engine with LLM(..., seed=process index) = 0 on one GPU, whatever the
# config; the engine gets TRAIN_SEED instead (TRAIN_SEED 0 = every earlier run).
import inspect
import trl.generation.vllm_generation as _vg
assert hasattr(_vg, "LLM") and "seed=" in inspect.getsource(_vg), "TRL's vLLM engine construction changed: check the seed"
if not getattr(_vg.LLM, "_train_seeded", False):
    class _SeededLLM(_vg.LLM):
        _train_seeded = True
        def __init__(self, *args, **kwargs):
            kwargs["seed"] = TRAIN_SEED
            print(f"[build] vLLM engine seed {TRAIN_SEED}")
            super().__init__(*args, **kwargs)
    _vg.LLM = _SeededLLM
trainer = JudgeGRPOTrainer(model=model, processing_class=tok, args=training_args, train_dataset=train_dataset,
                           reward_funcs=[_late(f) if isinstance(f, str) else f for f, _ in REWARD_FUNCS_TRAIN])
_bound.update(judge=make_judge_reward(trainer), step_monitor=make_step_monitor(trainer))
_pm = trainer.model
assert {"default", "ref"} <= set(_pm.peft_config), f"adapters: {list(_pm.peft_config)}"
assert all(p.requires_grad == (".default." in n) for n, p in _pm.named_parameters() if "lora_" in n), "a non-policy adapter trains"
_d = {n: p for n, p in _pm.named_parameters() if ".default." in n and "lora_" in n}
assert all(torch.equal(p, _pm.get_parameter(n.replace(".default.", ".ref."))) for n, p in _d.items()), "ref != S"
assert trainer._tokenize_prompts([train_dataset[0]["prompt"]])[0][0] == chat_ids(train_dataset[0]["prompt"]), \
    "the trainer's prompt tokens differ from chat_ids (chat_template_kwargs?)"
print(f"[adapters] {sorted(_pm.peft_config)}; policy trainable ({sum(p.numel() for p in _d.values()):,} params), ref = S; "
      "trainer prompt tokens = chat_ids")

trainer.vllm_generation.sync_weights()
_n = 0
for m in trainer.model.modules():
    if getattr(type(m).__call__, "__module__", "").startswith("vllm") and not hasattr(m, "do_not_compile"):
        m.do_not_compile = True
        _n += 1
print(f"[vLLM patch] {_n} module(s) marked do_not_compile")

def load_policy(path):
    """An adapter's weights into "default" (the policy), synced to vLLM."""
    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict
    res = set_peft_model_state_dict(trainer.model, load_file(f"{path}/adapter_model.safetensors"), adapter_name="default")
    assert not res.unexpected_keys, res.unexpected_keys[:4]
    trainer.vllm_generation.sync_weights()

def trainer_logps(ids_prompt, ids_completion, temperature):
    inp = torch.tensor([ids_prompt + ids_completion], device=trainer.model.device)
    with torch.no_grad():
        logits = trainer.model(input_ids=inp).logits[0, len(ids_prompt) - 1:-1].float() / temperature
    return torch.log_softmax(logits, -1).gather(1, inp[0, len(ids_prompt):, None])[:, 0].cpu().numpy()

SAMPLER_MAX_DIFF = 0.02
_ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for p in GRADE[4][:8]]
_outs = trainer.vllm_generation.llm.generate([{"prompt_token_ids": x} for x in _ids],
                                             SamplingParams(temperature=0.0, max_tokens=128, logprobs=1), use_tqdm=False)
_dd = []
for x, o in zip(_ids, _outs):
    t = list(o.outputs[0].token_ids)
    _dd.append(trainer_logps(x, t, 1.0) - np.array([lp[i].logprob for lp, i in zip(o.outputs[0].logprobs, t)]))
_seq = np.array([s.sum() for s in _dd])
_dd = np.concatenate(_dd)
print(f"[sampler check] {len(_dd)} tokens: mean |diff| {np.abs(_dd).mean():.4f}, signed {_dd.mean():+.4f}; "
      f"per-sequence log-ratio {np.round(_seq, 3).tolist()}")
assert np.abs(_dd).mean() < SAMPLER_MAX_DIFF and abs(_dd.mean()) < 0.01, "vLLM and the trainer disagree"

def generate_graded(puzzles_by_size, n, temperature, name):
    """The policy in the engine: n samples per puzzle (cap, seed SEED), labeled as S's grade file."""
    batch = [(s, p) for s in (4, 3) for p in puzzles_by_size[s]]
    ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for _, p in batch]
    outs = trainer.vllm_generation.llm.generate([{"prompt_token_ids": x} for x in ids],
                                                SamplingParams(n=n, temperature=temperature, top_p=1.0,
                                                               max_tokens=MAX_COMPLETION_LENGTH, seed=SEED), use_tqdm=False)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids))
    rows = [{"model": name, "n_numbers": s, "puzzle": p.get("puzzle", i), "numbers": json.dumps(p["numbers"]),
             "target": p["target"], "sample": k, "completion": c.text, "n_tokens": len(c.token_ids),
             "finished": c.finish_reason == "stop", "truncated": c.finish_reason != "stop"}
            for i, ((s, p), o) in enumerate(zip(batch, outs)) for k, c in enumerate(o.outputs)]
    return label(pd.DataFrame(rows))

print("[build] trainer ready")


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Capturing CUDA graphs (FULL): 100%|██████████| 19/19 [00:00<00:00, 26.04it/s]


[adapters] ['default', 'ref']; policy trainable (10,092,544 params), ref = S; trainer prompt tokens = chat_ids
[vLLM sync fix] streamed 196 LoRA layers as bf16(W + dW); training weights untouched
[vLLM patch] 0 module(s) marked do_not_compile
[sampler check] 1024 tokens: mean |diff| 0.0026, signed +0.0001; per-sequence log-ratio [-0.068, 0.284, 0.071, -0.061, -0.138, 0.075, -0.067, -0.015]
[build] trainer ready


In [10]:
# [judge] Consistency_Qwen_Judge_Countdown's judge into the trainer's model as a frozen adapter "judge". S's grade file at the cap (S_GRADE: N_GRADE
# puzzles per size x 8, T 0.9, as S's training rollouts; generated here if missing). Calibration (once, saved to
# CALIB_PATH), scored as the GRPO reward scores: weight = within-puzzle SD of the format reward (all completions) / of the
# judge score (scored completions); missing = the median judge score of the scored wrong completions. Self-judge check: within-puzzle win rate of correct over wrong (ties 1/2) for
# the judge, S itself (frozen "ref"), the untrained model and "shorter".
if "judge" not in trainer.model.peft_config:
    trainer.model.load_adapter(JUDGE_PATH, adapter_name="judge", is_trainable=False)
    trainer.model.set_adapter("default")
    _freeze_others(trainer.model)
assert all(p.requires_grad == (".default." in n) for n, p in trainer.model.named_parameters() if "lora_" in n)
if not os.path.exists(S_GRADE):
    load_policy(S_ADAPTER)
    os.makedirs(os.path.dirname(S_GRADE), exist_ok=True)
    generate_graded({s: v[:N_GRADE] for s, v in GRADE.items()}, GRADE_SAMPLES, TEMPERATURE, "sft").to_csv(S_GRADE, index=False)
    print(f"[judge] S graded: {S_GRADE}")
_cal_csv = CALIB_PATH.replace(".json", "_scores.csv")
if os.path.exists(CALIB_PATH):
    cal, ev = json.load(open(CALIB_PATH)), pd.read_csv(_cal_csv, keep_default_na=False)
    assert cal["judge"] == JUDGE_PATH and cal["reduction"] == JUDGE_REDUCTION, (cal["judge"], cal["reduction"])
    print(f"[judge] calibration loaded: {CALIB_PATH}")
else:
    t0 = time.time()
    ev = read_graded(S_GRADE)
    _gp = {(n, p["puzzle"]) for n, v in GRADE.items() for p in v}
    ev = ev[[(n, p) in _gp for n, p in zip(ev.n_numbers, ev.puzzle)]].reset_index(drop=True)
    texts = ev.completion.tolist()
    nums = [json.loads(x) for x in ev.numbers]
    items = [(n, int(t), judge_input(x)) for n, t, x in zip(nums, ev.target, texts)]
    ev["format_reward"] = format_total(texts)
    ev["correct_r"] = np.array(R.correctness_reward_func(None, texts, nums, ev.target.astype(int).tolist())) > 0
    ev["scored"] = [x is not None for _, _, x in items]
    for name, adapter in (("judge", "judge"), ("S", "ref"), ("base", None)):
        ev[f"{name}_mean"], ev[f"{name}_sum"] = judge_scores(trainer.model, items, adapter=adapter)
    ev["shorter"] = [-len(tok(x, add_special_tokens=False)["input_ids"]) if x else np.nan for _, _, x in items]
    sc = ev[ev.scored]
    sd_within = lambda d, col: float(np.sqrt(d.groupby(["n_numbers", "puzzle"])[col].var(ddof=0).mean()))
    key = f"judge_{JUDGE_REDUCTION}"
    cal = {"format_sd_within": sd_within(ev, "format_reward"), "judge_sd_within": sd_within(sc, key),
           "missing": float(sc.loc[~sc.correct_r, key].median()), "reduction": JUDGE_REDUCTION, "completions": len(ev),
           "scored": int(ev.scored.sum()), "judge": JUDGE_PATH, "source": S_GRADE}
    cal["weight"] = cal["format_sd_within"] / cal["judge_sd_within"]
    ev.drop(columns=["completion"]).to_csv(_cal_csv, index=False)
    json.dump(cal, open(CALIB_PATH, "w"), indent=1)
    print(f"[judge] calibrated on {len(ev)} completions in {(time.time() - t0) / 60:.1f} min; saved {CALIB_PATH}")
JUDGE_WEIGHT, JUDGE_MISSING = cal["weight"] * JUDGE_WEIGHT_SCALE, cal["missing"]
print(f"[judge] adapters {sorted(trainer.model.peft_config)} (only default trains); {JUDGE_REDUCTION}: weight "
      f"{JUDGE_WEIGHT:.4g} (format SD {cal['format_sd_within']:.3f} / judge SD {cal['judge_sd_within']:.3f}), missing "
      f"{JUDGE_MISSING:.1f}; scored {cal['scored']} of {cal['completions']}")
_rows = []
for size in (4, 3):
    x = ev[(ev.n_numbers == size) & (ev.scored.astype(str) == "True")]
    _k = (x.n_numbers.astype(str) + "-" + x.puzzle.astype(str)).tolist()
    _c = x.correct_r.astype(str).eq("True").values
    r = {"numbers": size, "scored": len(x), "correct": _c.mean()}
    for m in ("judge_sum", "judge_mean", "S_sum", "base_sum", "shorter"):
        r[m], r["pairs"] = within_puzzle_win_rate(_k, x[m].astype(float).values, _c)
    _rows.append(r)
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
    print(f"\n[self-judge check] S's grade file ({N_GRADE} per size x {GRADE_SAMPLES}, T {TEMPERATURE:g}, cap {MAX_COMPLETION_LENGTH}), "
          "scored completions: within-puzzle win rate of correct over wrong (ties 1/2; 0.5 = no signal)\n"
          + pd.DataFrame(_rows).to_string(index=False))


[judge] calibration loaded: /content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/dpo_empty_b0.1_lr2e-05_ep2/calibration_sum_missmed_S_grade_n200x8_t0.9_cap1024_seed0.json
[judge] adapters ['default', 'judge', 'ref'] (only default trains); sum: weight 0.05715 (format SD 0.824 / judge SD 14.418), missing -34.8; scored 1766 of 3200

[self-judge check] S's grade file (200 per size x 8, T 0.9, cap 1024), scored completions: within-puzzle win rate of correct over wrong (ties 1/2; 0.5 = no signal)
 numbers  scored  correct  judge_sum  pairs  judge_mean  S_sum  base_sum  shorter
       4     683    0.190      0.910    278       0.906  0.694     0.802    0.682
       3    1083    0.541      0.972   1120       0.941  0.825     0.851    0.807


In [11]:
# [train] Each arm in ARMS from S on the same puzzles and rollouts, one after the other on the same trainer: the policy
# adapter reset to S (or the arm's newest checkpoint: resume), a fresh optimizer and schedule, the judge's reward weight
# (calibrated in "judge", 0 in "format"), the arm's RUN_DIR for checkpoints, metrics and dumps. Held-out curve every
# CURVE_EVERY steps (step 0 = S, cached in S_DIR). At MAX_STEPS the final adapter goes to RUN_DIR/adapter and the arm is
# graded ([grade] functions below) before the next arm starts. An arm trained to MAX_STEPS is skipped. Rerunning a folder:
# its run.json settings are compared with these; a difference outside SAFE_TO_CHANGE asks before training on.
from safetensors.torch import load_file
from peft import set_peft_model_state_dict, get_peft_model_state_dict
sys.path.insert(0, f"{DRIVE}/code")
from runs import log_run                        # Consistency/runs.csv: one row per run start
from transformers import set_seed

class SeedCheck(TrainerCallback):
    """After step 1 (step 0's rollouts dumped): stop if they repeat the seed-0 run's (SEED0_STEP0), i.e. TRAIN_SEED did
    not reach the sampling."""
    def __init__(self, path):
        self.path, self.done = path, False
    def on_step_end(self, args, state, control, **kwargs):
        if self.done or not os.path.exists(self.path):
            return
        self.done = True
        resp = lambda f: [b.split("[Response]\n", 1)[1] for b in open(f).read().split("### Sample ")[1:]]
        same = sum(a == b for a, b in zip(resp(self.path), resp(SEED0_STEP0)))
        print(f"[seed check] step-0 rollouts identical to the seed-0 run: {same} of {len(resp(self.path))}")
        assert same < len(resp(self.path)) // 2, "TRAIN_SEED did not change the rollouts: stop and check [build]'s engine seed"

def grade_path(arm, step):
    return f"{RUN_DIRS[arm]}/grade/gen_{RUN_IDS[arm]}-step{step}.csv"

def grade_arm(arm, step):
    """The grade file of an arm at a step (generated with that checkpoint or the final adapter if missing)."""
    path = grade_path(arm, step)
    if not os.path.exists(path):
        load_policy(f"{RUN_DIRS[arm]}/adapter" if step == MAX_STEPS else f"{RUN_DIRS[arm]}/checkpoint-{step}")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        _g = {s: v[:N_GRADE] for s, v in GRADE.items()}
        generate_graded(_g, GRADE_SAMPLES, TEMPERATURE, os.path.basename(path)[4:-4]).to_csv(path, index=False)
        print(f"[grade] saved {path}")
    return read_graded(path)

def curve_eval(step, out_prefix):
    t0 = time.time()
    rows = {}
    for kind, n, T in (("greedy", 1, 0.0), ("sampled", CURVE_K, TEMPERATURE)):
        d = generate_graded(CURVE, n, T, f"step{step}")
        d.to_csv(f"{out_prefix}_{kind}.csv", index=False)
        for s, x in d.groupby("n_numbers"):
            ok = x.finished & x.correct
            rows[f"{kind} acc {s}"] = ok.mean()
            if kind == "sampled":
                rows[f"pass@{CURVE_K} {s}"] = x.assign(ok=ok).groupby("puzzle").ok.any().mean()
                rows[f"truncated {s}"] = 1 - x.finished.mean()
                rows[f"correct among answered {s}"] = ok.sum() / max((x.finished & x.answered).sum(), 1)
    return {"step": step, **rows, "minutes": (time.time() - t0) / 60}

class CurveEval(TrainerCallback):
    def __init__(self, path, out_dir):
        self.path, self.out_dir = path, out_dir
    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        rows = pd.read_csv(self.path).to_dict("records") if os.path.exists(self.path) else []
        if step % CURVE_EVERY or any(int(r["step"]) == step for r in rows):
            return
        trainer.vllm_generation.sync_weights()
        os.makedirs(self.out_dir, exist_ok=True)
        rows.append(curve_eval(step, f"{self.out_dir}/step{step:05d}"))
        pd.DataFrame(rows).to_csv(self.path, index=False)
        with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
            print(f"\n[curve {CUR['arm']}] held-out {CURVE_N} per size: greedy x 1, sampled x {CURVE_K} at T {TEMPERATURE:g}\n"
                  + pd.DataFrame(rows).to_string(index=False))

SAFE_TO_CHANGE = {"MAX_STEPS", "SAVE_STEPS", "DUMP_EVERY", "GRADE_ALSO", "MICRO_BATCH", "VLLM_GPU_MEMORY_UTILIZATION",
                  "UNASSIGN_WHEN_DONE", "ARMS", "RUN_IDS", "RUN_DIRS"}

_fn = lambda x: json.loads(re.sub(r"<function (\w+) at 0x[0-9a-f]+>", r"\1", json.dumps(x, default=lambda o: getattr(o, "__name__", str(o)))))

def run_config():
    """The settings saved in run.json: upper-case globals, functions by name, minus runtime state."""
    return _fn({k: v for k, v in globals().items() if k.isupper() and not k.startswith("_")
                and k not in ("GRADE", "CURVE", "SAFE_TO_CHANGE", "CUR")
                and isinstance(v, (int, float, str, bool, tuple, list, dict, type(None)))})

def check_config(arm, old):
    """A rerun of a folder: its saved settings vs these. Safe keys are printed; any other difference asks."""
    new = run_config()
    oc = {k: v for k, v in _fn(old.get("config", {})).items() if not k.startswith("_") and k != "CUR"}
    diff = {k: (oc.get(k, "(absent)"), new.get(k, "(absent)")) for k in set(oc) | set(new) if oc.get(k) != new.get(k)}
    if old.get("calibration") != cal:
        diff["calibration"] = (old.get("calibration"), cal)
    if oc.get("MAX_STEPS", 0) > MAX_STEPS:
        diff["MAX_STEPS lowered"] = (oc["MAX_STEPS"], MAX_STEPS)
    ask = {k: v for k, v in diff.items() if k not in SAFE_TO_CHANGE}
    for k, (a, b) in sorted(diff.items()):
        print(f"[train] {arm} setting {'CHANGED' if k in ask else 'changed'}: {k}: {str(a)[:150]} -> {str(b)[:150]}")
    if ask and input(f"[train] {arm}: continue {RUN_IDS[arm]} with the settings above? [y/N] ").strip().lower() != "y":
        raise RuntimeError(f"[train] {arm}: not continued; change RUN_TAG for a new run or restore the settings")
    return bool(diff)

class ResumeCheck(TrainerCallback):
    """The trainer's resume once restored step, optimizer and scheduler but not the policy adapter (Llama, 2026-10-21):
    the adapter is compared with the checkpoint's and loaded if they differ."""
    def __init__(self, ckpt):
        self.ckpt = ckpt
    def on_train_begin(self, args, state, control, **kwargs):
        want = load_file(f"{self.ckpt}/adapter_model.safetensors")
        now = get_peft_model_state_dict(trainer.model, adapter_name="default")
        diff = max((now[k].detach().float().cpu() - want[k].float()).abs().max().item() for k in want)
        if diff > 0:
            res = set_peft_model_state_dict(trainer.model, want, adapter_name="default")
            assert not res.unexpected_keys, res.unexpected_keys[:4]
        trainer.vllm_generation.sync_weights()
        print(f"[resume check] step {state.global_step}: policy vs {os.path.basename(self.ckpt)} max |diff| {diff:g}"
              + (" -> loaded" if diff > 0 else ""))

def reset_for_arm(arm):
    """Fresh optimizer, schedule, TRL step state and callbacks; the policy = S; the arm's reward weight and RUN_DIR."""
    rd = RUN_DIRS[arm]
    CUR.update(arm=arm, run_dir=rd)
    trainer.args.output_dir = rd
    trainer.optimizer, trainer.lr_scheduler = None, None
    trainer._step, trainer._buffered_inputs, trainer._last_loaded_step = 0, None, -1
    for v in trainer._metrics.values():
        v.clear()
    trainer.reward_weights[JUDGE_IDX] = JUDGE_WEIGHT if arm == "judge" else 0.0
    trainer.callback_handler.callbacks = [c for c in trainer.callback_handler.callbacks
                                          if type(c).__name__ not in ("CSVLoggerCallback", "CurveEval", "ResumeCheck")]
    trainer.add_callback(CSVLoggerCallback(f"{rd}/metrics.csv"))
    trainer.add_callback(CurveEval(f"{rd}/curve_n34x{CURVE_N}.csv", f"{rd}/curve"))
    load_policy(S_ADAPTER)
    want = load_file(f"{S_ADAPTER}/adapter_model.safetensors")
    now = get_peft_model_state_dict(trainer.model, adapter_name="default")
    assert max((now[k].detach().float().cpu() - want[k].float()).abs().max().item() for k in want) == 0, "policy != S"
    assert all(p.requires_grad == (".default." in n) for n, p in trainer.model.named_parameters() if "lora_" in n)
    print(f"[train] {arm}: policy = S, judge weight {float(trainer.reward_weights[JUDGE_IDX]):.4g}, RUN_DIR {rd}")

# step 0 of the curve: S (both arms)
if not os.path.exists(f"{CURVE_S}_row.json"):
    load_policy(S_ADAPTER)
    os.makedirs(os.path.dirname(CURVE_S), exist_ok=True)
    json.dump(curve_eval(0, CURVE_S), open(f"{CURVE_S}_row.json", "w"), indent=1)
_s_row = json.load(open(f"{CURVE_S}_row.json"))

for arm in ARMS:
    rd = RUN_DIRS[arm]
    _old = json.load(open(f"{rd}/run.json")) if os.path.exists(f"{rd}/run.json") else None
    _changed = check_config(arm, _old) if _old is not None else False
    if trained_to(rd) == MAX_STEPS:
        print(f"[train] {arm}: trained to {MAX_STEPS}, skipped")
    else:
        reset_for_arm(arm)
        _cp = f"{rd}/curve_n34x{CURVE_N}.csv"
        if not os.path.exists(_cp):
            pd.DataFrame([_s_row]).to_csv(_cp, index=False)
        _ck = sorted(glob.glob(f"{rd}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
        if _ck:
            trainer.add_callback(ResumeCheck(_ck[-1]))
        _hist = (_old.get("history", []) + [{k: _old.get(k) for k in ("created", "finished", "config", "calibration")}]
                 if _changed else (_old or {}).get("history", []))
        json.dump({"id": RUN_IDS[arm], "kind": "grpo", "notebook": "Consistency_GRPO_Qwen_Countdown",
                   "parents": {"model": MODEL, "start": S_ADAPTER, "judge": JUDGE_PATH if arm == "judge" else None},
                   "created": (_old or {}).get("created", datetime.now().isoformat()), "calibration": cal,
                   "config": run_config(), "history": _hist,
                   "status": "training"}, open(f"{rd}/run.json", "w"), indent=1, default=str)
        print(f"[train] {arm}: {'resuming from ' + os.path.basename(_ck[-1]) if _ck else 'from S'}")
        log_run("Consistency_GRPO_Qwen_Countdown",
                f"GRPO {arm} arm (judge vs format control), SFT start, cap {MAX_COMPLETION_LENGTH}"
                + (f", resumed from {os.path.basename(_ck[-1])}" if _ck else ""), MODEL, rd)
        if not _ck:                              # a resume restores the checkpoint's RNG state instead
            set_seed(TRAIN_SEED)
            if TRAIN_SEED and os.path.exists(SEED0_STEP0):
                trainer.add_callback(SeedCheck(f"{rd}/step_00000.txt"))
        _t0 = time.time()
        trainer.train(resume_from_checkpoint=_ck[-1] if _ck else None)
        print(f"[train] {arm}: {(time.time() - _t0) / 60:.1f} min")
        trainer.model.save_pretrained(f"{rd}/adapter", selected_adapters=["default"])
        if os.path.isdir(f"{rd}/adapter/default"):
            for f in os.listdir(f"{rd}/adapter/default"):
                shutil.move(f"{rd}/adapter/default/{f}", f"{rd}/adapter/{f}")
            os.rmdir(f"{rd}/adapter/default")
        _r = json.load(open(f"{rd}/run.json"))
        _r.update(status="trained", finished=datetime.now().isoformat())
        json.dump(_r, open(f"{rd}/run.json", "w"), indent=1, default=str)
    for st in (MAX_STEPS, *GRADE_ALSO):
        grade_arm(arm, st)
print("[train] all arms trained and graded")


[train] judge: policy = S, judge weight 0.05715, RUN_DIR /content/drive/MyDrive/Consistency/countdown/grpo/qwen3-0.6b-sft-judge-w1-median
[train] judge: from S


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


[runs] logged /content/drive/MyDrive/Consistency/countdown/grpo/qwen3-0.6b-sft-judge-w1-median in /content/drive/MyDrive/Consistency/runs.csv
[judge step 0] correct 0.266 (4: 0.062, 3: 0.469) | answered 0.55 | correct among answered 0.479 | judge sum -29.405, win 1.00 (298 pairs)


Step,Training Loss
5,0.407584
10,0.418442
15,0.383682
20,0.372602
25,0.285796
30,0.224919
35,0.068452
40,0.132992
45,0.060756
50,0.135935


[judge step 1] correct 0.094 (4: 0.000, 3: 0.188) | answered 0.54 | correct among answered 0.174 | judge sum -33.549, win 0.97 (146 pairs)
[judge step 2] correct 0.234 (4: 0.078, 3: 0.391) | answered 0.51 | correct among answered 0.462 | judge sum -31.397, win 0.97 (252 pairs)
[judge step 3] correct 0.289 (4: 0.094, 3: 0.484) | answered 0.69 | correct among answered 0.420 | judge sum -29.363, win 0.97 (323 pairs)
[judge step 4] correct 0.250 (4: 0.047, 3: 0.453) | answered 0.55 | correct among answered 0.457 | judge sum -29.998, win 0.94 (258 pairs)
[judge 5] reward=-0.569, no-gradient groups=0.00, grad_norm=0.109, kl=0.0001, length=667, truncated=0.433, correct=0.227, judge sum=-27.641
[judge step 5] correct 0.281 (4: 0.125, 3: 0.438) | answered 0.61 | correct among answered 0.462 | judge sum -31.452, win 0.95 (342 pairs)
[judge step 6] correct 0.227 (4: 0.125, 3: 0.328) | answered 0.70 | correct among answered 0.326 | judge sum -29.652, win 1.00 (267 pairs)
[judge step 7] correct 0.2

Step,Training Loss
5,0.407584
10,0.418442
15,0.383682
20,0.372602
25,0.285796
30,0.224919
35,0.068452
40,0.132992
45,0.060756
50,0.135935


[judge step 103] correct 0.273 (4: 0.125, 3: 0.422) | answered 0.99 | correct among answered 0.276 | judge sum -21.145, win 0.94 (157 pairs)
[judge step 104] correct 0.219 (4: 0.172, 3: 0.266) | answered 1.00 | correct among answered 0.219 | judge sum -20.702, win 1.00 (158 pairs)
[judge 105] reward=0.883, no-gradient groups=0.00, grad_norm=0.039, kl=0.0356, length=253, truncated=0.002, correct=0.341, judge sum=-19.673
[judge step 105] correct 0.273 (4: 0.234, 3: 0.312) | answered 1.00 | correct among answered 0.273 | judge sum -20.337, win 0.82 (297 pairs)
[judge step 106] correct 0.211 (4: 0.078, 3: 0.344) | answered 1.00 | correct among answered 0.211 | judge sum -21.177, win 0.94 (155 pairs)
[judge step 107] correct 0.281 (4: 0.172, 3: 0.391) | answered 0.99 | correct among answered 0.283 | judge sum -19.585, win 0.93 (164 pairs)
[judge step 108] correct 0.188 (4: 0.062, 3: 0.312) | answered 0.99 | correct among answered 0.189 | judge sum -21.266, win 0.92 (172 pairs)
[judge step 1

In [12]:
# [compare] S's grade file (S_GRADE: same puzzles, 8 x T 0.9, at the cap, seed 0) and each arm's grades:
# the table per size, then per-puzzle accuracy differences (arm - S, and judge - format at the same step) with 95%
# bootstrap CIs over puzzles.
S_g = read_graded(S_GRADE)
G = {("S", 0): S_g}
for arm in ARMS:
    for st in (*GRADE_ALSO, MAX_STEPS):
        if os.path.exists(grade_path(arm, st)):
            G[(arm, st)] = read_graded(grade_path(arm, st))
tab = pd.DataFrame([{"policy": a, "step": st, "numbers": s, **stats(x)} for (a, st), d in G.items()
                    for s, x in d.groupby("n_numbers")]).sort_values(["numbers", "step", "policy"])
tab.to_csv(f"{S_DIR}/compare_{'-'.join(RUN_IDS.values())[:150]}.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
    print(f"[compare] S's grade puzzles, {N_GRADE} x {GRADE_SAMPLES} per size, T {TEMPERATURE:g}, cap {MAX_COMPLETION_LENGTH}\n"
          + tab.to_string(index=False))
_rng = np.random.default_rng(SEED)

def diff_ci(a, b):
    """Per-puzzle accuracy a - b (same puzzles), mean and 95% bootstrap."""
    acc = lambda d: d.assign(ok=d.finished & d.correct).groupby(["n_numbers", "puzzle"]).ok.mean()
    out = []
    for s in (4, 3):
        dd = (acc(a[a.n_numbers == s]) - acc(b[b.n_numbers == s])).dropna().values
        bs = dd[_rng.integers(0, len(dd), (10000, len(dd)))].mean(1)
        out.append(f"{s}-number {dd.mean():+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    return "; ".join(out)

for (a, st), d in G.items():
    if a != "S":
        print(f"{a} step {st} - S: {diff_ci(d, S_g)}")
if {"format", "judge"} <= set(ARMS):
    for st in (*GRADE_ALSO, MAX_STEPS):
        if ("judge", st) in G and ("format", st) in G:
            print(f"judge - format, step {st}: {diff_ci(G[('judge', st)], G[('format', st)])}")


[compare] S's grade puzzles, 200 x 8 per size, T 0.9, cap 1024
policy  step  numbers    n  accuracy  format ok  solution parses  truncated  answered wrong  correct among answered  tokens (median, finished)  puzzles: correct + answered wrong  pass@k
     S     0        3 1600     0.367      0.685            0.649      0.315           0.318                   0.536                    354.500                                162   0.880
 judge    50        3 1600     0.364      0.991            0.985      0.009           0.627                   0.367                    170.000                                130   0.690
 judge   100        3 1600     0.504      0.998            0.996      0.001           0.495                   0.504                    170.500                                105   0.710
 judge   150        3 1600     0.561      0.987            0.986      0.012           0.427                   0.568                    237.000                                110   0.755
     S 

In [ ]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE (a failed cell stops Run all before this).
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()
